# Pipeline Sentinel-2 SR (Colab) — Jussara-GO

Este notebook monta o Google Drive, prepara pastas, autentica o Earth Engine, define a AOI de Jussara-GO, gera composições mensais 2024–2025, calcula índices (NDVI e EVI opcional), exporta GeoTIFFs e cria uma tabela parquet de séries temporais por polígono.

In [ ]:
# ====== VARIÁVEIS GERAIS (ajuste aqui) ======
PROJECT_NAME = "doutorado-geoprocessamento"
DRIVE_MOUNT = "/content/drive"
PROJECT_DIR = f"{DRIVE_MOUNT}/MyDrive/{PROJECT_NAME}"
DATA_DIR = f"{PROJECT_DIR}/data"
EXPORT_DIR = f"{PROJECT_DIR}/exports"
VECTORS_DIR = f"{PROJECT_DIR}/vectors"

# AOI: use Asset (se informar) ou limite administrativo
AOI_ASSET_ID = ""  # Ex: 'users/seu_usuario/aoi_jussara'
AOI_ADMIN_SOURCE = "FAO/GAUL/2015/level2"
AOI_ADMIN_FILTERS = {"ADM0_NAME": "Brazil", "ADM1_NAME": "Goias", "ADM2_NAME": "Jussara"}


EE_PROJECT = ""  # Ex: 'seu-projeto-ee' (obrigatório para muitos logins)

REQUIRE_EE_PROJECT = False  # coloque True se sua conta exigir project
# Intervalo de tempo e parâmetros
START_DATE = "2024-01-01"
END_DATE = "2025-12-31"
CLOUD_FILTER = 50  # % de nuvens máximas (metadado)
SCALE = 10
CRS = "EPSG:4326"

# Índices a exportar
EXPORT_NDVI = True
EXPORT_EVI = True  # opcional
EXPORT_PREFIX = "JUSSARA_S2SR"

# Vetor para séries temporais (GeoPackage/Shapefile no Drive)
POLYGONS_VECTOR_PATH = f"{VECTORS_DIR}/poligonos.gpkg"  # ajuste conforme necessário
PARQUET_OUTPUT_PATH = f"{PROJECT_DIR}/timeseries.parquet"

print("✅ Variáveis definidas")


In [ ]:
from google.colab import drive
import os

drive.mount(DRIVE_MOUNT)

for path in [PROJECT_DIR, DATA_DIR, EXPORT_DIR, VECTORS_DIR]:
    os.makedirs(path, exist_ok=True)

print("📁 Pastas prontas:")
print("-", PROJECT_DIR)
print("-", DATA_DIR)
print("-", EXPORT_DIR)
print("-", VECTORS_DIR)


In [ ]:
%pip -q install earthengine-api geemap geopandas rasterio pandas

import ee
import geemap
import geopandas as gpd
import rasterio
import pandas as pd
import numpy as np
import glob
import re

print("✅ Pacotes instalados e importados")


In [ ]:
ee.Authenticate()
if not EE_PROJECT and REQUIRE_EE_PROJECT:
    raise ValueError("Defina EE_PROJECT nas variáveis gerais para evitar o erro de projeto do Earth Engine.")
try:
    if EE_PROJECT:
        ee.Initialize(project=EE_PROJECT)
    else:
        ee.Initialize()
except Exception as exc:
    if not EE_PROJECT:
        raise ValueError("Falha ao inicializar o Earth Engine sem project. Preencha EE_PROJECT e tente novamente.") from exc
    raise
print("✅ Earth Engine autenticado e inicializado")


In [ ]:
# Definição da AOI
if AOI_ASSET_ID:
    aoi = ee.FeatureCollection(AOI_ASSET_ID)
    aoi_label = "asset"
else:
    admin = ee.FeatureCollection(AOI_ADMIN_SOURCE)
    aoi = admin
    for key, value in AOI_ADMIN_FILTERS.items():
        aoi = aoi.filter(ee.Filter.eq(key, value))
    aoi_label = "admin"

aoi_geom = aoi.geometry()
print(f"✅ AOI definida via {aoi_label}")
print("📌 Número de features:", aoi.size().getInfo())

Map = geemap.Map()
Map.centerObject(aoi, 9)
Map.addLayer(aoi, {}, "AOI")
Map


In [ ]:
# Pipeline Sentinel-2 SR com máscara de nuvem
def mask_s2_sr(image):
    qa = image.select("QA60")
    scl = image.select("SCL")
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).And(qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    # Remove sombras, nuvens e pixels inválidos no SCL
    scl_mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11))
    return image.updateMask(mask).updateMask(scl_mask).copyProperties(image, ["system:time_start"])

def add_indices(image):
    ndvi = image.normalizedDifference(["B8", "B4"]).rename("NDVI")
    evi = image.expression(
        "2.5 * ((NIR - RED) / (NIR + 6 * RED - 7.5 * BLUE + 1))",
        {"NIR": image.select("B8"), "RED": image.select("B4"), "BLUE": image.select("B2")}
    ).rename("EVI")
    return image.addBands([ndvi, evi])

collection = (
    ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
    .filterBounds(aoi_geom)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.lte("CLOUDY_PIXEL_PERCENTAGE", CLOUD_FILTER))
    .map(mask_s2_sr)
    .map(add_indices)
)

print("✅ Coleção filtrada:", collection.size().getInfo())

# Composições mensais
def month_composite(year, month):
    start = ee.Date.fromYMD(year, month, 1)
    end = start.advance(1, "month")
    monthly = collection.filterDate(start, end)
    composite = monthly.median()
    return composite.set({"year": year, "month": month, "system:time_start": start.millis()})

years = list(range(2024, 2026))
months = list(range(1, 13))
images = []
for y in years:
    for m in months:
        images.append(month_composite(y, m))
monthly_collection = ee.ImageCollection.fromImages(images)

print("✅ Composições mensais criadas:", monthly_collection.size().getInfo())


In [ ]:
# Exportação para Google Drive
bands_to_export = []
if EXPORT_NDVI:
    bands_to_export.append("NDVI")
if EXPORT_EVI:
    bands_to_export.append("EVI")

print("✅ Bandas para exportar:", bands_to_export)

tasks = []
monthly_list = monthly_collection.toList(monthly_collection.size())
for i in range(monthly_collection.size().getInfo()):
    img = ee.Image(monthly_list.get(i)).select(bands_to_export)
    year = ee.Number(img.get("year")).format().getInfo()
    month = ee.Number(img.get("month")).format().getInfo().zfill(2)
    for band in bands_to_export:
        description = f"{EXPORT_PREFIX}_{band}_{year}_{month}"
        task = ee.batch.Export.image.toDrive(
            image=img.select([band]),
            description=description,
            folder=os.path.basename(EXPORT_DIR),
            fileNamePrefix=description,
            region=aoi_geom,
            scale=SCALE,
            crs=CRS,
            maxPixels=1e13,
        )
        task.start()
        tasks.append(task)
        print("🚀 Export iniciado:", description)

print("✅ Total de exports iniciados:", len(tasks))


In [ ]:
# Leitura dos GeoTIFFs exportados e geração de parquet
# Aguarde o término das exports antes de rodar esta célula

tif_paths = sorted(glob.glob(f"{EXPORT_DIR}/*.tif"))
print("📄 GeoTIFFs encontrados:", len(tif_paths))

if len(tif_paths) == 0:
    print("⚠️ Nenhum GeoTIFF encontrado. Verifique se as exports finalizaram.")
else:
    gdf = gpd.read_file(POLYGONS_VECTOR_PATH)
    if gdf.empty:
        raise ValueError("O arquivo de polígonos está vazio.")

    records = []
    date_pattern = re.compile(r"(NDVI|EVI)_(\d{4})_(\d{2})")

    for tif in tif_paths:
        with rasterio.open(tif) as src:
            raster_crs = src.crs
            gdf_proj = gdf.to_crs(raster_crs)
            data = src.read(1, masked=True)

            match = date_pattern.search(os.path.basename(tif))
            if match:
                index_name, year, month = match.group(1), match.group(2), match.group(3)
                date = f"{year}-{month}-01"
            else:
                index_name, date = "INDEX", ""

            for idx, row in gdf_proj.iterrows():
                geom = [row.geometry]
                out_image, out_transform = rasterio.mask.mask(src, geom, crop=True)
                out_data = out_image[0]
                masked = np.ma.masked_equal(out_data, src.nodata)
                mean_val = float(masked.mean()) if masked.count() > 0 else np.nan

                records.append({
                    "polygon_id": row.get("id", idx),
                    "date": date,
                    "index": index_name,
                    "mean": mean_val,
                })

    df = pd.DataFrame(records)
    df.to_parquet(PARQUET_OUTPUT_PATH, index=False)
    print("✅ Parquet gerado em:", PARQUET_OUTPUT_PATH)
    print(df.head())
